## Analysis of downloaded data

In [ ]:
import pydicom

In [ ]:
img = pydicom.dcmread(r"../data\test\5\1.2.826.0.1.3680043.8.498.12506063821850171756494207689001728484\1.2.826.0.1.3680043.8.498.12754854321177212940363159597270003304\1.2.826.0.1.3680043.8.498.72533800876543798738969965510832915095-c.dcm")

In [6]:
img_ = img.pixel_array

In [7]:
img_.shape

(256, 256)

In [11]:
img_.max(), img_.min()

(23221, 8321)

In [12]:
import os

In [14]:
import numpy as np

In [ ]:
sizes = np.zeros((1743,2))
i = 0
for root, dirs, files in os.walk("../data/train"):
    for file in files:
        if file.endswith('.dcm'):
            img = pydicom.dcmread(os.path.join(root, file)).pixel_array
            sizes[i,:] = img.shape
            i+=1
            print(f"Completed {i}/1743", end='\r')

Completed 1738/1743

In [20]:
sizes

array([[256., 256.],
       [256., 256.],
       [256., 256.],
       ...,
       [  0.,   0.],
       [  0.,   0.],
       [  0.,   0.]])

In [22]:
np.unique(sizes)

array([  0., 256.])

In [25]:
r"dataset\unifesp-x-ray-body-part-classifier.archive"

'dataset\\unifesp-x-ray-body-part-classifier.archive'

In [ ]:
img = pydicom.dcmread(r"../dataset\train\train\1\1.2.826.0.1.3680043.8.498.89102450329340531816015855773961083133\1.2.826.0.1.3680043.8.498.11278653404499913987623237519434199794\1.2.826.0.1.3680043.8.498.65452424240994805812717428674475343109-c.dcm")


In [27]:
img_ = img.pixel_array

In [28]:
img_.shape

(3480, 4240)

In [29]:
import pandas as pd

In [ ]:
df = pd.read_csv("../dataset/train.csv")

In [31]:
df

,SOPInstanceUID,Target
0,1.2.826.0.1.3680043.8.498.10025629581362719970...,0
1,1.2.826.0.1.3680043.8.498.10036150326276641158...,15
2,1.2.826.0.1.3680043.8.498.10038426859954986240...,12
3,1.2.826.0.1.3680043.8.498.10050991192143676483...,14
4,1.2.826.0.1.3680043.8.498.10053309524595490852...,3
...,...,...
1733,1.2.826.0.1.3680043.8.498.99711016467406172920...,14
1734,1.2.826.0.1.3680043.8.498.99727280485333900933...,11
1735,1.2.826.0.1.3680043.8.498.99935008698914755885...,3
1736,1.2.826.0.1.3680043.8.498.99969671777553755311...,2


In [41]:
def norm(x):
    min_ = x.min()
    return (x - min_)/(x.max()-min_)

## Preprocess 
Preprocess the images and save them into smaller files for faster processing
Warning: this takes more than an hour on my laptop.

In [ ]:
import time
from PIL import Image
count = 1
import os
if not os.path.exists('../dataset/preprocessed'):
    os.mkdir('../dataset/preprocessed')
if not os.path.exists('../dataset/preprocessed/train'):
    os.mkdir('../dataset/preprocessed/train')

time_estimate = 0
mean_time = 0

for root_dir, cur_dir, files in os.walk("../dataset/train"):
    for file in files:
        if file.endswith('.dcm'):
            print(f"Working on train preprocessing {count:0>4}/{1738}. Time remaining={int(time_estimate//60)}:{int(time_estimate%60//1):0>2}", end='\r')
            output_path = os.path.join("../dataset", "preprocessed", "train", file.removesuffix(".dcm"))
            start = time.time()
            
            if not os.path.exists(output_path):
                img = pydicom.dcmread(os.path.join(root_dir, file)).pixel_array
                load_time = time.time()-start
                float_img = norm(img)
                smaller = np.array(Image.fromarray(float_img).resize((224,224)))
                normed = (smaller - smaller.mean()) / smaller.std()
                low = np.percentile(normed, 1)
                high = np.percentile(normed, 99)
                clipped = np.clip(normed, low, high)
                output = (clipped - low) / (high - low)
                np.save(output_path, output)
            
            end = time.time()
            
            mean_time = (end - start) + mean_time * count
            count += 1
            mean_time = mean_time / count
            time_estimate = (1738 - count) * mean_time
print()

Working on train 1738/1738. Time remaining=0:001


## Create a torch dataset file
Used for faster and easier training with pytorch.

In [ ]:
data_df = pd.DataFrame({"SOPInstanceUID":pd.Series(dtype='str'), "data":pd.Series(dtype='object')})
preprocessed_dir = '../dataset/preprocessed/train'
for fp in os.listdir(preprocessed_dir):
    row_df = pd.DataFrame({"SOPInstanceUID":fp.removesuffix("-c.npy"), "data":[np.load(f"{preprocessed_dir}/{fp}")]})
    data_df = pd.concat([data_df,row_df])

In [56]:
import torch
class XrayImageDataset(torch.utils.data.Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

In [59]:
joined_df = pd.merge(df, data_df)
whole_dataset = XrayImageDataset(joined_df['data'], joined_df['Target'])

In [74]:
# How many targets labels there are for each sample. 
joined_df['Target'].apply(lambda x: len(x.strip().split(' '))).value_counts()

Target
1    1606
2     122
3      10
Name: count, dtype: int64

In [78]:
# Select only the first label since some of them are multi label
joined_df['label'] = joined_df['Target'].apply(lambda x: x.strip().split(' ')[0])

In [80]:
data_tensor = torch.tensor(joined_df['data'].array)
label_tensor = torch.tensor((joined_df['label'].apply(lambda x: int(x))).array)

C:\Users\Mitch\AppData\Local\Temp\ipykernel_31052\1251880674.py:1: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:257.)
  data_tensor = torch.tensor(joined_df['data'].array)


In [ ]:
dataset_data = {
    'features':data_tensor,
    'labels':label_tensor
}

torch.save(dataset_data, '../dataset/processed_dataset.pt')